In [4]:
from pathlib import Path
import json

result_path = Path("../output_vfc")

json_files = list(result_path.glob('**/RunnerResult_DefaultRefiner.json'))
data = []
for file in json_files:
    try:
        datapoint = json.load(open(file))
        data.append(datapoint)
    except Exception as e:
        print(file)
len(data)

23

In [5]:
import pandas as pd

rows = []
for run in data:
    id = run['baseDir'].replace('/app/output/', '').replace('_', ':')
    perf = run.get("performanceTracker", {})

    for metric, events in perf.items():
        if not isinstance(events, list):
            continue
        for idx, ev in enumerate(events, start=1):
            rows.append({
                "id": id,
                "metric": metric,
                "attempt": idx,
                "start_time": ev.get("startTime"),
                "duration_ms": ev.get("duration"),
            })

perf_df = pd.DataFrame(rows)

# Optional convenience column
if not perf_df.empty:
    perf_df["duration_s"] = perf_df["duration_ms"] / 1000.0

perf_df

,id,metric,attempt,start_time,duration_ms,duration_s
0,SNYK-JS-TREEKIT-1077068,codeql.init,1,1769594810007,10405,10.405
1,SNYK-JS-TREEKIT-1077068,getExportsFromPackage,1,1769594820412,6495,6.495
2,SNYK-JS-TREEKIT-1077068,model.query,1,1769594826927,1433,1.433
3,SNYK-JS-TREEKIT-1077068,model.query,2,1769594828367,557,0.557
4,SNYK-JS-TREEKIT-1077068,model.query,3,1769594828963,1562,1.562
...,...,...,...,...,...,...
1437,npm:moment:20170905,model.query,170,1769596382424,12341,12.341
1438,npm:moment:20170905,model.query,171,1769596394806,12808,12.808
1439,npm:moment:20170905,model.query,172,1769596407651,0,0.000
1440,npm:moment:20170905,codeql.analyse,1,1769594939316,50323,50.323


In [6]:
# Build a runtime summary by id, then append experiment-level totals
runtime_by_id_df = (
    perf_df.groupby("id", as_index=False)["duration_ms"]
    .sum()
    .rename(columns={"duration_ms": "total_runtime_ms"})
)

total_runtime_ms = runtime_by_id_df["total_runtime_ms"].sum()
num_ids = runtime_by_id_df["id"].nunique()
avg_runtime_per_id_ms = total_runtime_ms / num_ids if num_ids else 0

summary_rows_df = pd.DataFrame([
    {"id": "__TOTAL_EXPERIMENT__", "total_runtime_ms": total_runtime_ms},
    {"id": "__AVG_PER_ID__", "total_runtime_ms": avg_runtime_per_id_ms},
])

runtime_summary_df = pd.concat([runtime_by_id_df, summary_rows_df], ignore_index=True)

# Keep seconds for numeric analysis and add a human-readable duration string
runtime_summary_df["total_runtime_s"] = runtime_summary_df["total_runtime_ms"] / 1000.0
runtime_summary_df["total_runtime_human"] = pd.to_timedelta(
    runtime_summary_df["total_runtime_ms"], unit="ms"
).astype(str)

# Drop the millisecond column from final display
runtime_summary_df = runtime_summary_df.drop(columns=["total_runtime_ms"])

runtime_summary_df

,id,total_runtime_s,total_runtime_human
0,SNYK-JS-CHRONONODE-1083228,621.167000,0 days 00:10:21.167000
1,SNYK-JS-DOTOBJECT-548905,689.232000,0 days 00:11:29.232000
2,SNYK-JS-FASTCSV-1049538,5065.129000,0 days 01:24:25.129000
3,SNYK-JS-FASTJSONPATCH-595663,346.376000,0 days 00:05:46.376000
4,SNYK-JS-HTMLPARSESTRINGIFY-1079306,328.389000,0 days 00:05:28.389000
5,SNYK-JS-HTMLPARSESTRINGIFY2-1079307,728.736000,0 days 00:12:08.736000
6,SNYK-JS-KILLPORT-1078535,396.573000,0 days 00:06:36.573000
7,SNYK-JS-LIBNMAP-72551,5162.503000,0 days 01:26:02.503000
8,SNYK-JS-MARKDOWNIT-2331914,666.642000,0 days 00:11:06.642000
9,SNYK-JS-MATHJS-1016401,4073.536000,0 days 01:07:53.536000
